# FAPE — Student Performance Stage 2: ThresholdOptimizer
## Education Domain — Sex Fairness Intervention

**Stage 2:** Fairlearn ThresholdOptimizer on Student Performance Math and Portuguese.
**Key finding:** Sex gap reverses between subjects. Small dataset challenge — subgroups too small for reliable estimation.
**Note:** ThresholdOptimizer non-deterministic in fairlearn 0.13.0.
**Sensitive attributes:** Sex (F/M), Age

In [1]:
import sys, os, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from student_loader import load_student_performance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
os.makedirs('../figures/stage2', exist_ok=True)

MODELS = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

def prepare_student(data, sensitive_col):
    X=data['X'].copy(); y=data['y'].copy()
    sensitive=X[sensitive_col].astype(str).copy()
    for col in X.select_dtypes(include=['object','category']).columns:
        le=LabelEncoder(); X[col]=le.fit_transform(X[col].astype(str))
    X=X.fillna(0).values
    y=pd.to_numeric(y,errors='coerce').fillna(0).astype(int).values
    return X,y,sensitive

def run_threshold(model,X_tr,y_tr,X_te,y_te,s_tr,s_te,constraint):
    try:
        to=ThresholdOptimizer(estimator=model,constraints=constraint,
                              predict_method='auto',objective='balanced_accuracy_score')
        to.fit(X_tr,y_tr,sensitive_features=s_tr)
        yp=to.predict(X_te,sensitive_features=s_te)
        return {'acc':accuracy_score(y_te,yp),'f1':f1_score(y_te,yp,zero_division=0),
                'dpd':demographic_parity_difference(y_te,yp,sensitive_features=s_te),
                'eod':equalized_odds_difference(y_te,yp,sensitive_features=s_te),'pred':yp}
    except: return None

st=load_student_performance(); subjects=list(st.keys())
all_results={}
for subj in subjects:
    data=st[subj]
    X,y,sex=prepare_student(data,'sex')
    X_train,X_test,y_train,y_test,idx_train,idx_test=train_test_split(
        X,y,np.arange(len(y)),test_size=0.2,random_state=42,stratify=y)
    scaler=StandardScaler()
    X_train_sc=scaler.fit_transform(X_train); X_test_sc=scaler.transform(X_test)
    sex_train=sex.iloc[idx_train].reset_index(drop=True)
    sex_test=sex.iloc[idx_test].reset_index(drop=True)
    baseline={}
    for name,model in MODELS.items():
        m=model.__class__(**model.get_params())
        if name=='LogisticRegression':
            m.fit(X_train_sc,y_train); yp=m.predict(X_test_sc); X_tr,X_te=X_train_sc,X_test_sc
        else:
            m.fit(X_train,y_train); yp=m.predict(X_test); X_tr,X_te=X_train,X_test
        baseline[name]={'acc':accuracy_score(y_test,yp),'f1':f1_score(y_test,yp,zero_division=0),
            'dpd':demographic_parity_difference(y_test,yp,sensitive_features=sex_test),
            'eod':equalized_odds_difference(y_test,yp,sensitive_features=sex_test),
            'model':m,'X_tr':X_tr,'X_te':X_te}
    dp={n:run_threshold(r['model'],r['X_tr'],y_train,r['X_te'],y_test,sex_train,sex_test,'demographic_parity') for n,r in baseline.items()}
    eo={n:run_threshold(r['model'],r['X_tr'],y_train,r['X_te'],y_test,sex_train,sex_test,'equalized_odds') for n,r in baseline.items()}
    all_results[subj]={'baseline':baseline,'dp':dp,'eo':eo,'sex_test':sex_test,'y_test':y_test,
                       'X_test':X_test,'X_test_sc':X_test_sc,'y_train':y_train,'n':len(data['X'])}
models=list(MODELS.keys()); short=['LR','RF','GB']
x=np.arange(len(models)); width=0.25
print('Setup complete --', subjects)

  Downloaded: student-mat (395 rows), student-por (649 rows)
  ✓ student_math: 395 rows | 30 features | positive rate: 0.410 | median grade: 11.0
  ✓ student_portuguese: 649 rows | 30 features | positive rate: 0.425 | median grade: 12.0


Setup complete -- ['math', 'portuguese']


## 1. Accuracy vs Fairness by Subject

Portuguese has higher accuracy. Math has lower accuracy due to smaller sample.

In [2]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    base_accs=[res['baseline'][m]['acc'] for m in models]
    dp_accs=[res['dp'][m]['acc'] if res['dp'].get(m) else 0 for m in models]
    eo_accs=[res['eo'][m]['acc'] if res['eo'].get(m) else 0 for m in models]
    ax.bar(x-width,base_accs,width,label='Baseline',color='#95a5a6',edgecolor='white')
    ax.bar(x,dp_accs,width,label='DP Constraint',color='#3498db',edgecolor='white')
    ax.bar(x+width,eo_accs,width,label='EO Constraint',color='#e74c3c',edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(short)
    ax.set_title(f'{subj.capitalize()} (n={all_results[subj]["n"]})',fontsize=11,fontweight='bold')
    ax.set_ylabel('Accuracy'); ax.legend(); ax.set_ylim(0.4,1.0)
plt.suptitle('Student Performance -- Accuracy: Baseline vs Constrained',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_accuracy_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 1 saved -- student_accuracy_comparison.png')

Fig 1 saved -- student_accuracy_comparison.png


## 2. DPD Before vs After DP Constraint

Sex gap reduction by subject. LR most responsive.

In [3]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    base_dpds=[res['baseline'][m]['dpd'] for m in models]
    dp_dpds=[res['dp'][m]['dpd'] if res['dp'].get(m) else 0 for m in models]
    w2=0.35
    ax.bar(x-w2/2,base_dpds,w2,label='Baseline DPD',color='#e74c3c',edgecolor='white')
    ax.bar(x+w2/2,dp_dpds,w2,label='After DP Constraint',color='#3498db',edgecolor='white')
    for i,(b,a) in enumerate(zip(base_dpds,dp_dpds)):
        imp=(b-a)/b*100 if b!=0 else 0
        ax.annotate(f'{imp:+.0f}%',xy=(i,max(b,a)+0.01),ha='center',fontsize=9,color='green' if imp>0 else 'red')
    ax.set_xticks(x); ax.set_xticklabels(short)
    ax.set_title(f'{subj.capitalize()} -- DPD Before vs After',fontsize=11,fontweight='bold')
    ax.set_ylabel('DPD'); ax.legend()
plt.suptitle('Student Performance -- Sex Fairness: DPD Before vs After DP Constraint',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_dpd_before_after.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 2 saved -- student_dpd_before_after.png')

Fig 2 saved -- student_dpd_before_after.png


## 3. EOD Before vs After EO Constraint

Equalized odds improvement by subject.

In [4]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    base_eods=[res['baseline'][m]['eod'] for m in models]
    eo_eods=[res['eo'][m]['eod'] if res['eo'].get(m) else 0 for m in models]
    w2=0.35
    ax.bar(x-w2/2,base_eods,w2,label='Baseline EOD',color='#e74c3c',edgecolor='white')
    ax.bar(x+w2/2,eo_eods,w2,label='After EO Constraint',color='#2ecc71',edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(short)
    ax.set_title(f'{subj.capitalize()} -- EOD Before vs After',fontsize=11,fontweight='bold')
    ax.set_ylabel('EOD'); ax.legend()
plt.suptitle('Student Performance -- Sex Fairness: EOD Before vs After EO Constraint',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_eod_before_after.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 3 saved -- student_eod_before_after.png')

Fig 3 saved -- student_eod_before_after.png


## 4. Sex Prediction Rates

True vs predicted rate by sex. Sex gap direction reverses between Math and Portuguese.

In [5]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    gb=res['baseline']['GradientBoosting']
    yp_base=gb['model'].predict(res['X_test'])
    sex_te=res['sex_test']; y_te=res['y_test']
    true_rates=[y_te[sex_te==s].mean() if (sex_te==s).sum()>0 else 0 for s in ['F','M']]
    base_rates=[yp_base[sex_te==s].mean() if (sex_te==s).sum()>0 else 0 for s in ['F','M']]
    x4=np.arange(2); w4=0.35
    ax.bar(x4-w4/2,true_rates,w4,label='True Rate',color='#2ecc71',edgecolor='white')
    ax.bar(x4+w4/2,base_rates,w4,label='GB Predicted',color='#95a5a6',edgecolor='white')
    for i,(t,b) in enumerate(zip(true_rates,base_rates)):
        ax.text(i-w4/2,t+0.01,f'{t:.2f}',ha='center',fontsize=9)
        ax.text(i+w4/2,b+0.01,f'{b:.2f}',ha='center',fontsize=9)
    ax.set_xticks(x4); ax.set_xticklabels(['Female','Male'])
    ax.set_title(f'{subj.capitalize()} -- True vs Predicted by Sex',fontsize=11,fontweight='bold')
    ax.set_ylabel('Positive Rate'); ax.legend()
plt.suptitle('Student Performance -- Sex Gap: True Rate vs GB Predicted Rate',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_sex_prediction_rates.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 4 saved -- student_sex_prediction_rates.png')

Fig 4 saved -- student_sex_prediction_rates.png


## 5. F1 Score Comparison

F1 stability across baseline and constrained models.

In [6]:
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    base_f1s=[res['baseline'][m]['f1'] for m in models]
    dp_f1s=[res['dp'][m]['f1'] if res['dp'].get(m) else 0 for m in models]
    eo_f1s=[res['eo'][m]['f1'] if res['eo'].get(m) else 0 for m in models]
    ax.bar(x-width,base_f1s,width,label='Baseline',color='#95a5a6',edgecolor='white')
    ax.bar(x,dp_f1s,width,label='DP Constraint',color='#3498db',edgecolor='white')
    ax.bar(x+width,eo_f1s,width,label='EO Constraint',color='#e74c3c',edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(short)
    ax.set_title(f'{subj.capitalize()} -- F1 Comparison',fontsize=11,fontweight='bold')
    ax.set_ylabel('F1 Score'); ax.legend(); ax.set_ylim(0,1.0)
plt.suptitle('Student Performance -- F1 Score: Baseline vs Constrained',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_f1_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 5 saved -- student_f1_comparison.png')

Fig 5 saved -- student_f1_comparison.png


## 6. DIR by Sex — EEOC 80% Compliance

All DIR below EEOC 0.8 threshold. Sex gap direction reverses between subjects.

In [7]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,subj in zip(axes,subjects):
    res=all_results[subj]
    sex_te=res['sex_test']; y_te=res['y_test']
    gb=res['baseline']['GradientBoosting']
    yp=gb['model'].predict(res['X_test'])
    dirs={}
    for s in ['F','M']:
        mask=sex_te==s
        if mask.sum()>0: dirs[s]=yp[mask].mean()
    dir_fm=dirs['F']/dirs['M'] if 'F' in dirs and 'M' in dirs and dirs['M']>0 else 0
    dir_mf=dirs['M']/dirs['F'] if 'F' in dirs and 'M' in dirs and dirs['F']>0 else 0
    bars=ax.bar(['F/M DIR','M/F DIR'],[dir_fm,dir_mf],color=['#3498db','#e74c3c'],edgecolor='white')
    ax.axhline(0.8,color='orange',linestyle='--',linewidth=2,label='EEOC 0.8 threshold')
    for bar,val in zip(bars,[dir_fm,dir_mf]):
        ax.text(bar.get_x()+bar.get_width()/2,val+0.01,f'{val:.3f}',ha='center',fontsize=10,fontweight='bold')
    ax.set_title(f'{subj.capitalize()} -- DIR by Sex (EEOC 80% rule)',fontsize=11,fontweight='bold')
    ax.set_ylabel('Disparate Impact Ratio'); ax.legend(); ax.set_ylim(0,1.5)
plt.suptitle('Student Performance -- DIR by Sex: EEOC 80% Compliance Check',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/stage2/student_dir_by_sex.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 6 saved -- student_dir_by_sex.png')

Fig 6 saved -- student_dir_by_sex.png


## 7. Key Findings

**Small dataset challenge:** Math n=395, Portuguese n=649 — subgroup sizes make metric estimation noisy
**Sex gap reverses between subjects** — confirmed in Stage 2 (known from EDA)
**LR DP constraint most responsive** — reduces DPD significantly in both subjects
**All DIR below EEOC 0.8** — ThresholdOptimizer applied to improve compliance
**ThresholdOptimizer non-deterministic** in fairlearn 0.13.0 — direction consistent across runs